In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset
import os
# from datasets import Audio
from IPython.display import Audio
os.environ["HF_HOME"] = "/home/lenovoai/kelvin/speech_ai/data/hf_home"
os.environ["HF_DATASETS_CACHE"] = "/home/lenovoai/kelvin/speech_ai/data/hf_datasets"


device = "cpu"
torch_dtype = torch.float16

from datasets import load_dataset
ds = load_dataset("openslr/librispeech_asr", "clean", split="validation")
print(ds)


/home/lenovoai/kelvin/agent_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating validation split: 100%|██████████| 2703/2703 [00:00<00:00, 13442.42 examples/s]

Dataset({
    features: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'],
    num_rows: 2703
})


In [3]:
model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)
processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=torch_dtype,
    device=device,
)

Loading weights: 100%|██████████| 1259/1259 [00:00<00:00, 14482.47it/s]


In [9]:
ds['text'][0]

"HE WAS IN A FEVERED STATE OF MIND OWING TO THE BLIGHT HIS WIFE'S ACTION THREATENED TO CAST UPON HIS ENTIRE FUTURE"

In [22]:


dec = ds['audio'][6]               # torchcodec AudioDecoder
samples = dec.get_all_samples()
audio_array = samples.data.squeeze(0).cpu().numpy()
sampling_rate = samples.sample_rate

Audio(audio_array, rate=sampling_rate)    # must be the LAST expression in the cell


In [23]:
# Run through your existing pipeline
result = pipe(
    {"array": audio_array, "sampling_rate": sampling_rate},
    return_timestamps=True,
    generate_kwargs={"language": "english", "task": "transcribe"},
)
print(result["text"])

 He could arrange that satisfactorily, for Carrie would be glad to wait, if necessary.


In [4]:
import io
import soundfile as sf
from datasets import load_dataset, Audio

dataset = load_dataset(
    "distil-whisper/librispeech_long", "clean", split="validation"
).cast_column("audio", Audio(decode=False))   # disables torchcodec


In [ ]:
raw = dataset[0]["audio"]                     # {"path": ..., "bytes": ...}
array, sampling_rate = sf.read(io.BytesIO(raw["bytes"]))

print(array.shape, sampling_rate)


(999280,) 16000


In [11]:
result=pipe(array, sampling_rate=sampling_rate)

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.9.1+cu130) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call